In [ ]:
import warnings
from pathlib import Path

import numpy as np
import prism

from imagematerials.concepts import create_building_graph, create_vehicle_graph
from imagematerials.factory import ModelFactory, Namespace
from imagematerials.model import GenericMaterials, GenericStocks, MaterialIntensities
from imagematerials.util import export_to_netcdf, import_from_netcdf, rebroadcast_prep_data
from imagematerials.vehicles import (
    preprocess,
)


In [2]:
def get_vema_prep():
    base_dir = "../data/raw"
    prep_fp = Path("prep_vema.nc")
    if not prep_fp.is_file():
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            orig_prep_data = preprocess(base_dir)
        export_to_netcdf(orig_prep_data, prep_fp)

    prep_data = import_from_netcdf(prep_fp)
    share_coords = set()
    for cur_type in prep_data["shares"].Type.values:
        share_coords.add(cur_type.split(" - ")[0])
    output_coords_type = [x for x in prep_data["stocks"].Type.values if x not in share_coords] + list(prep_data["shares"].coords["Type"].values)
    prep_data.pop("shares")
    knowledge_graph = create_vehicle_graph()
    new_prep_data = rebroadcast_prep_data(prep_data, knowledge_graph, dim="Type", output_coords=output_coords_type)
    region_coords = np.sort(prep_data["stocks"].coords["Region"].values.astype(int)).astype(str)[:-2]
    new_prep_data = rebroadcast_prep_data(new_prep_data, knowledge_graph, dim="Region", output_coords=region_coords)
    new_prep_data["knowledge_graph"] = knowledge_graph

    new_prep_data["weights"] = new_prep_data.pop("vehicle_weights")
    ns_vhc = Namespace("vhc", new_prep_data)
    return ns_vhc

In [3]:
def get_buildings_prep():
    base_dir = Path("..", "IMAGE-Mat_old_version", "IMAGE-Mat", "BUMA")
    prep_fp = Path("prep_buildings.nc")
    if not prep_fp.is_file():
        prep_data = preprocess(base_dir)
        export_to_netcdf(prep_data, prep_fp)
    else:
        prep_data = import_from_netcdf(prep_fp)
    new_prep_data = {k: v for k, v in prep_data.items()}
    new_prep_data["knowledge_graph"] = create_building_graph()
    ns_bld = Namespace("bld", new_prep_data)
    return ns_bld

In [4]:
name_spaces = [get_vema_prep(), get_buildings_prep()]
ns_coordinates = {
    "Region": name_spaces[0].coordinates["Region"],
    "material": list(set(name_spaces[0].coordinates["material"] + name_spaces[1].coordinates["material"]))
}
ns_combined = Namespace("combi", {}, coordinates=ns_coordinates)
name_spaces.append(ns_combined)

In [6]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [14]:
REGION = prism.Dimension("Region")
MATERIAL_TYPE = prism.Dimension("material")

@prism.interface
class SumMaterials(prism.Model):
    Region: prism.Coords[REGION]
    material: prism.Coords[MATERIAL_TYPE]

    input_data: tuple[str] = ("inflow_materials", )
    output_data: tuple[str] = ("summed_inflow_materials", )



    summed_inflow_materials: prism.TimeVariable[REGION, MATERIAL_TYPE, "count"] = prism.export()

    def compute_initial_values(self, time: prism.Timeline):
        pass

    def compute_values(self, time: prism.Time, inflow_materials):
        t, dt = time.t, time.dt
        self.summed_inflow_materials[t].loc[:] = 0
        for inflow in inflow_materials:
            self.summed_inflow_materials[t].loc[{"material": inflow[t].coords["material"]}] += inflow[t].sum("Type")

In [15]:
factory = ModelFactory(
    name_spaces, complete_timeline
    ).add(GenericStocks, ["bld", "vhc"]
    ).add(GenericMaterials,  "vhc"
    ).add(MaterialIntensities, "bld",
    ).add(SumMaterials, "combi", input_sources={"inflow_materials": ["vhc", "bld"]}
    )
model = factory.finish()

In [16]:
import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(simulation_timeline)

In [17]:
model.combi["summed_inflow_materials"][2020]

<xarray.DataArray (Region: 26, material: 18)> Size: 4kB
<Quantity([[1.27609877e+07 2.49731139e+01 0.00000000e+00 1.23812853e+07
  0.00000000e+00 0.00000000e+00 0.00000000e+00 1.18997035e+08
  0.00000000e+00 8.45956745e+03 0.00000000e+00 1.18463638e+05
  1.85624658e+08 0.00000000e+00 1.75275480e+09 0.00000000e+00
  1.30805371e+08 3.14053541e+04]
 [3.36564733e+08 1.76238028e+02 0.00000000e+00 4.28875683e+08
  0.00000000e+00 0.00000000e+00 2.67188064e+01 3.23067501e+09
  0.00000000e+00 5.87994803e+04 0.00000000e+00 2.12016975e+06
  5.97381451e+09 0.00000000e+00 5.50736600e+10 0.00000000e+00
  4.04450670e+09 1.26120988e+05]
 [4.88578189e+07 1.87623275e+01 0.00000000e+00 1.18287609e+07
  0.00000000e+00 0.00000000e+00 0.00000000e+00 4.06405599e+08
  0.00000000e+00 6.57689128e+03 0.00000000e+00 5.78159698e+04
  3.06233775e+08 0.00000000e+00 2.12529865e+09 0.00000000e+00
  1.44455118e+08 2.41481618e+04]
 [3.60967848e+07 1.00467599e+01 0.00000000e+00 3.13046942e+07
  0.00000000e+00 0.00000000e+00 0.00000000e+00 2.86972004e+08
  0.00000000e+00 3.52176177e+03 0.00000000e+00 1.43820057e+05
  6.09546924e+08 0.00000000e+00 4.85035939e+09 0.00000000e+00
  3.77215884e+08 1.28002039e+04]
...
 [2.61241602e+08 1.00300513e+02 0.00000000e+00 4.03063689e+08
  1.93740914e+05 0.00000000e+00 0.00000000e+00 1.49960783e+09
  0.00000000e+00 2.49599996e+04 7.81951814e+06 3.10987442e+07
  1.13682511e+09 3.66474139e+07 5.79906299e+09 0.00000000e+00
  3.14691297e+08 7.25211039e+04]
 [2.06758568e+07 1.41121574e+01 0.00000000e+00 8.36536630e+06
  0.00000000e+00 0.00000000e+00 0.00000000e+00 1.61443639e+08
  0.00000000e+00 9.82962960e+03 0.00000000e+00 1.71350736e+04
  2.11803731e+08 0.00000000e+00 1.39287592e+09 0.00000000e+00
  1.02882241e+08 3.05630521e+04]
 [1.91296195e+08 1.28646570e+02 0.00000000e+00 3.95999238e+07
  0.00000000e+00 0.00000000e+00 3.49003417e-01 1.65709359e+09
  0.00000000e+00 3.74134862e+04 0.00000000e+00 6.25315619e+04
  1.02542084e+09 0.00000000e+00 6.94751312e+09 0.00000000e+00
  4.46822113e+08 1.19138897e+05]
 [9.66508377e+07 3.83090432e+01 0.00000000e+00 7.01811707e+07
  2.64156930e+04 0.00000000e+00 3.35001547e+00 7.15495464e+08
  0.00000000e+00 1.08650826e+04 1.06615576e+06 4.31580571e+06
  5.25970343e+08 4.99670832e+06 3.30601678e+09 0.00000000e+00
  2.06477099e+08 3.39531728e+04]], 'count')>
Coordinates:
  * Region    (Region) <U2 208B '1' '2' '3' '4' '5' ... '22' '23' '24' '25' '26'
  * material  (material) <U9 648B 'Glass' 'Copper' 'Mn' ... 'Rubber' 'Concrete'